In [140]:
import dotenv
import os

dotenv.load_dotenv(dotenv_path='.env')

True

In [141]:
import pyarrow as pa
import pandas
from pyiceberg.catalog.rest import RestCatalog
from pyiceberg.exceptions import NamespaceAlreadyExistsError
from pyiceberg.transforms import IdentityTransform, DayTransform

In [232]:
# 1. Connect to R2 Data Catalog

# Define catalog connection details
ENV = "dev"
CATALOG = "unprice-lakehouse-" + ENV
ACCOUNT_ID  = os.environ['ACCOUNT_ID']
CATALOG_URI = "https://catalog.cloudflarestorage.com/" + ACCOUNT_ID + "/" + CATALOG
TOKEN       = os.environ['TOKEN']
WAREHOUSE   = ACCOUNT_ID + "_" + CATALOG

# Connect to R2 Data Catalog
catalog = RestCatalog(
    name="unprice",
    warehouse=WAREHOUSE,
    uri=CATALOG_URI,
    token=TOKEN,
)

# list namespaces
print(catalog.list_namespaces())

[('lakehouse',)]


In [233]:
# drop tables
# catalog.drop_table('lakehouse.metadata')
# catalog.drop_table('lakehouse.entitlement_snapshot')
# catalog.drop_table('lakehouse.usage')
# catalog.drop_table('lakehouse.verification')

In [234]:
# List tables in the "default" namespace
tables = catalog.list_tables(namespace='lakehouse')
print(tables)

[('lakehouse', 'usage'), ('lakehouse', 'metadata'), ('lakehouse', 'verification'), ('lakehouse', 'entitlement_snapshot')]


In [235]:
# Load and print schema for every table in the lakehouse namespace
tables = catalog.list_tables(namespace='lakehouse')
for table_id in tables:
    table = catalog.load_table(table_id)
    print(f"--- {table_id[0]}.{table_id[1]} ---")
    print(table.schema())
    print()

--- lakehouse.usage ---
table {
  1: __ingest_ts: required timestamp
  2: event_date: required timestamp
  3: id: required string
  4: request_id: required string
  5: project_id: required string
  6: customer_id: required string
  7: timestamp: required timestamp
  8: idempotence_key: required string
  9: entitlement_id: required string
  10: feature_slug: required string
  11: schema_version: required int
  12: usage: optional double
  13: allowed: optional boolean
  14: deleted: optional long
  15: meta_id: optional string
  16: country: optional string
  17: region: optional string
  18: action: optional string
  19: key_id: optional string
  20: unit_of_measure: optional string
  21: cost: optional double
  22: rate_amount: optional double
  23: rate_currency: optional string
}

--- lakehouse.metadata ---
table {
  1: __ingest_ts: required timestamp
  2: event_date: required timestamp
  3: id: required string
  4: project_id: required string
  5: customer_id: required string
  6: 

In [236]:
table.refresh()  # <-- this is the key
df = table.scan().to_arrow()
print(df)

pyarrow.Table
__ingest_ts: timestamp[us] not null
event_date: timestamp[us] not null
id: string not null
project_id: string not null
customer_id: string not null
schema_version: int32 not null
timestamp: timestamp[us] not null
feature_slug: string not null
feature_type: string not null
unit_of_measure: string not null
reset_config: string
aggregation_method: string not null
limit: int64
effective_at: timestamp[us] not null
expires_at: timestamp[us]
version: string
grants: string
metadata: string
----
__ingest_ts: [[2026-02-22 00:28:43.000000]]
event_date: [[2026-02-20 00:00:00.000000]]
id: [["ent_11TqNF9PdDUQ4i4dVXH7fT"]]
project_id: [["proj_11STWG6AokEni2F3eQugHb"]]
customer_id: [["cus_11TqNF6bCebUjnx55pk6vs"]]
schema_version: [[1]]
timestamp: [[2026-02-20 21:54:40.841000]]
feature_slug: [["events"]]
feature_type: [["usage"]]
unit_of_measure: [["units"]]
...


In [237]:
# List and peek contents of every table in the lakehouse namespace
import pandas as pd

tables = catalog.list_tables(namespace='lakehouse')
MAX_ROWS = 1000  # rows to show per table

for table_id in tables:
    name = f"{table_id[0]}.{table_id[1]}"
    table = catalog.load_table(table_id)
    scan = table.scan(limit=MAX_ROWS)
    df = scan.to_pandas()
    print(f"\n=== {name} (showing up to {MAX_ROWS} rows) ===")
    if df.empty:
        print("(no rows)")
    else:
        display(df)


=== lakehouse.usage (showing up to 1000 rows) ===


,__ingest_ts,event_date,id,request_id,project_id,customer_id,timestamp,idempotence_key,entitlement_id,feature_slug,...,deleted,meta_id,country,region,action,key_id,unit_of_measure,cost,rate_amount,rate_currency
0,2026-02-22 00:28:43,2026-02-22,01KJ1BXK59EKEMHPWM7R200HAW,req_11U6adkii4TVffgcm4jcEJ,proj_11STWG6AokEni2F3eQugHb,cus_11TqNF6bCebUjnx55pk6vs,2026-02-22 00:27:48.263,123e4567-e89b-12d3-a456-426614174000:177172006...,ent_11TqNF9PdDUQ4i4dVXH7fT,events,...,0,15330311789501757000,DE,TXL,create,api_11STWG6JzNRqo1QeChEWuC,unit,0.00,0.0,EUR
1,2026-02-22 00:28:43,2026-02-22,01KJ1BXJYY8WMDG3APQENVNYGG,req_11U6adjrNQNWs7pcUVgqCk,proj_11STWG6AokEni2F3eQugHb,cus_11TqNF6bCebUjnx55pk6vs,2026-02-22 00:27:48.060,123e4567-e89b-12d3-a456-426614174000:177172006...,ent_11TqNF9PdDUQ4i4dVXH7fT,events,...,0,15330311789501757000,DE,TXL,create,api_11STWG6JzNRqo1QeChEWuC,unit,0.00,0.0,EUR
2,2026-02-22 00:28:43,2026-02-22,01KJ1BXJRVM5GSPYYEFB22QYCG,req_11U6adj2GBygYvpCmjQsQs,proj_11STWG6AokEni2F3eQugHb,cus_11TqNF6bCebUjnx55pk6vs,2026-02-22 00:27:47.866,123e4567-e89b-12d3-a456-426614174000:177172006...,ent_11TqNF9PdDUQ4i4dVXH7fT,events,...,0,15330311789501757000,DE,TXL,create,api_11STWG6JzNRqo1QeChEWuC,unit,0.00,0.0,EUR
3,2026-02-22 00:28:43,2026-02-22,01KJ1BXJE5T0M96EG4KV68SA41,req_11U6adhZCydXuiSgJR9D8p,proj_11STWG6AokEni2F3eQugHb,cus_11TqNF6bCebUjnx55pk6vs,2026-02-22 00:27:47.523,123e4567-e89b-12d3-a456-426614174000:177172006...,ent_11TqNF9PdDUQ4i4dVXH7fT,events,...,0,15330311789501757000,DE,TXL,create,api_11STWG6JzNRqo1QeChEWuC,unit,0.00,0.0,EUR
4,2026-02-22 00:28:43,2026-02-22,01KJ1BXJ8RHKJ498MSFMD8K542,req_11U6adgpJoB8BPd31ndo7a,proj_11STWG6AokEni2F3eQugHb,cus_11TqNF6bCebUjnx55pk6vs,2026-02-22 00:27:47.350,123e4567-e89b-12d3-a456-426614174000:177172006...,ent_11TqNF9PdDUQ4i4dVXH7fT,events,...,0,15330311789501757000,DE,TXL,create,api_11STWG6JzNRqo1QeChEWuC,unit,0.00,0.0,EUR
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66,2026-02-22 00:28:43,2026-02-22,01KJ1BEFMHM3VX8665SQX34VEX,req_11U6a1Fptdbud4TXRuk6Jk,proj_11STWG6AokEni2F3eQugHb,cus_11TqNF6bCebUjnx55pk6vs,2026-02-22 00:19:33.133,123e4567-e89b-12d3-a456-426614174000:177171957...,ent_11TqNF9PdDUQ4i4dVXH7fT,events,...,0,15330311789501757000,DE,TXL,create,api_11STWG6JzNRqo1QeChEWuC,unit,0.00,0.0,EUR
67,2026-02-22 00:28:43,2026-02-22,01KJ1BEFDXNBMCBVZF0CD7BWH5,req_11U6a1EvZv1yWtFDznSaVK,proj_11STWG6AokEni2F3eQugHb,cus_11TqNF6bCebUjnx55pk6vs,2026-02-22 00:19:32.922,123e4567-e89b-12d3-a456-426614174000:177171957...,ent_11TqNF9PdDUQ4i4dVXH7fT,events,...,0,15330311789501757000,DE,TXL,create,api_11STWG6JzNRqo1QeChEWuC,unit,0.00,0.0,EUR
68,2026-02-22 00:28:43,2026-02-22,01KJ1BEF7WPKWNKVSME2PRN2DX,req_11U6a1E6i5p2MhMzUerNrR,proj_11STWG6AokEni2F3eQugHb,cus_11TqNF6bCebUjnx55pk6vs,2026-02-22 00:19:32.729,123e4567-e89b-12d3-a456-426614174000:177171957...,ent_11TqNF9PdDUQ4i4dVXH7fT,events,...,0,15330311789501757000,DE,TXL,create,api_11STWG6JzNRqo1QeChEWuC,unit,0.01,0.0,EUR
69,2026-02-22 00:28:43,2026-02-22,01KJ1BEF1NFB5Y1PJS6GAVTWEC,req_11U6a1DEsC6z4741V6cSsj,proj_11STWG6AokEni2F3eQugHb,cus_11TqNF6bCebUjnx55pk6vs,2026-02-22 00:19:32.528,123e4567-e89b-12d3-a456-426614174000:177171957...,ent_11TqNF9PdDUQ4i4dVXH7fT,events,...,0,15330311789501757000,DE,TXL,create,api_11STWG6JzNRqo1QeChEWuC,unit,0.00,0.0,EUR



=== lakehouse.metadata (showing up to 1000 rows) ===


,__ingest_ts,event_date,id,project_id,customer_id,schema_version,timestamp,payload
0,2026-02-22 00:28:44,2026-02-22,15330311789501757000,proj_11STWG6AokEni2F3eQugHb,cus_11TqNF6bCebUjnx55pk6vs,1,2026-02-22 00:27:48.263,"""{\""projectId\"":\""xc\"",\""resourceId\"":\""sdf\"",..."
1,2026-02-21 15:15:32,2026-02-21,11286326239215174000,proj_11STWG6AokEni2F3eQugHb,cus_11TqNF6bCebUjnx55pk6vs,1,2026-02-21 15:14:31.331,"""{\""projectId\"":\""123\"",\""resourceId\"":\""123\""..."



=== lakehouse.verification (showing up to 1000 rows) ===


,__ingest_ts,event_date,id,project_id,entitlement_id,feature_slug,customer_id,request_id,timestamp,schema_version,...,region,meta_id,action,key_id,usage,remaining,unit_of_measure,cost,rate_amount,rate_currency
0,2026-02-21 21:34:58,2026-02-21,777,proj_11STWG6AokEni2F3eQugHb,ent_11TqNF9PdDUQ4i4dVXH7fT,events,cus_11TqNF6bCebUjnx55pk6vs,req_11U6MNpVz9nKPck3TQNcm9,2026-02-21 21:33:56.979,1,...,TXL,11286326239215174000,read,api_11STWG6JzNRqo1QeChEWuC,100.0,867,None,NaN,NaN,None
1,2026-02-21 21:34:58,2026-02-21,778,proj_11STWG6AokEni2F3eQugHb,ent_11TqNF9PdDUQ4i4dVXH7fT,events,cus_11TqNF6bCebUjnx55pk6vs,req_11U6MNvcqaE8JADDYQwtxv,2026-02-21 21:33:58.410,1,...,TXL,11286326239215174000,read,api_11STWG6JzNRqo1QeChEWuC,100.0,867,None,NaN,NaN,None
2,2026-02-21 21:34:58,2026-02-21,779,proj_11STWG6AokEni2F3eQugHb,ent_11TqNF9PdDUQ4i4dVXH7fT,events,cus_11TqNF6bCebUjnx55pk6vs,req_11U6MNxf83iKoJeLGs6Pwb,2026-02-21 21:33:58.887,1,...,TXL,11286326239215174000,read,api_11STWG6JzNRqo1QeChEWuC,100.0,867,None,NaN,NaN,None
3,2026-02-21 21:34:58,2026-02-21,780,proj_11STWG6AokEni2F3eQugHb,ent_11TqNF9PdDUQ4i4dVXH7fT,events,cus_11TqNF6bCebUjnx55pk6vs,req_11U6MNyWUB3WTHZTXn48YW,2026-02-21 21:33:59.086,1,...,TXL,11286326239215174000,read,api_11STWG6JzNRqo1QeChEWuC,100.0,867,None,NaN,NaN,None
4,2026-02-21 21:34:58,2026-02-21,781,proj_11STWG6AokEni2F3eQugHb,ent_11TqNF9PdDUQ4i4dVXH7fT,events,cus_11TqNF6bCebUjnx55pk6vs,req_11U6MNzM59pCrkXLiMeyA5,2026-02-21 21:33:59.282,1,...,TXL,11286326239215174000,read,api_11STWG6JzNRqo1QeChEWuC,100.0,867,None,NaN,NaN,None
5,2026-02-21 15:15:32,2026-02-21,749,proj_11STWG6AokEni2F3eQugHb,ent_11TqNF9PdDUQ4i4dVXH7fT,events,cus_11TqNF6bCebUjnx55pk6vs,req_11U5rSfaStoUiC4QUfStB9,2026-02-21 15:14:31.331,1,...,TXL,11286326239215174000,read,api_11STWG6JzNRqo1QeChEWuC,100.0,867,None,NaN,NaN,None
6,2026-02-21 15:15:32,2026-02-21,750,proj_11STWG6AokEni2F3eQugHb,ent_11TqNF9PdDUQ4i4dVXH7fT,events,cus_11TqNF6bCebUjnx55pk6vs,req_11U5rSgKqqdUJnGYcqB3vz,2026-02-21 15:14:31.506,1,...,TXL,11286326239215174000,read,api_11STWG6JzNRqo1QeChEWuC,100.0,867,None,NaN,NaN,None
7,2026-02-21 15:15:32,2026-02-21,751,proj_11STWG6AokEni2F3eQugHb,ent_11TqNF9PdDUQ4i4dVXH7fT,events,cus_11TqNF6bCebUjnx55pk6vs,req_11U5rShDfnqZ3b17qi7g2u,2026-02-21 15:14:31.715,1,...,TXL,11286326239215174000,read,api_11STWG6JzNRqo1QeChEWuC,100.0,867,None,NaN,NaN,None
8,2026-02-21 15:15:32,2026-02-21,752,proj_11STWG6AokEni2F3eQugHb,ent_11TqNF9PdDUQ4i4dVXH7fT,events,cus_11TqNF6bCebUjnx55pk6vs,req_11U5rShxpMUxSNq6YXd6gn,2026-02-21 15:14:31.889,1,...,TXL,11286326239215174000,read,api_11STWG6JzNRqo1QeChEWuC,100.0,867,None,NaN,NaN,None
9,2026-02-21 15:15:32,2026-02-21,753,proj_11STWG6AokEni2F3eQugHb,ent_11TqNF9PdDUQ4i4dVXH7fT,events,cus_11TqNF6bCebUjnx55pk6vs,req_11U5rSipfFBwQ8yc9PGn7S,2026-02-21 15:14:32.090,1,...,TXL,11286326239215174000,read,api_11STWG6JzNRqo1QeChEWuC,100.0,867,None,NaN,NaN,None



=== lakehouse.entitlement_snapshot (showing up to 1000 rows) ===


,__ingest_ts,event_date,id,project_id,customer_id,schema_version,timestamp,feature_slug,feature_type,unit_of_measure,reset_config,aggregation_method,limit,effective_at,expires_at,version,grants,metadata
0,2026-02-22 00:28:43,2026-02-20,ent_11TqNF9PdDUQ4i4dVXH7fT,proj_11STWG6AokEni2F3eQugHb,cus_11TqNF6bCebUjnx55pk6vs,1,2026-02-20 21:54:40.841,events,usage,units,"{""name"":""monthly"",""resetInterval"":""month"",""res...",sum,1000,2026-02-13 23:36:24.309,NaT,Vj/FJUu11dRFTRM0V7w9qhODfRS3w1qE4rdZSrnXr4s=,"[{""id"":""grnt_11TqNF6z1qrTz4rKewJMwJ"",""type"":""s...","{""realtime"":false,""notifyUsageThreshold"":95,""o..."


In [230]:
from pyiceberg.expressions import And, EqualTo

# Load your table
table = catalog.load_table("lakehouse.verification")  # adjust namespace.table_name
table.refresh()

seven_days_ago = (datetime.utcnow() - timedelta(days=7)).isoformat()

# Build the expression tree
filter_expr = And(
    GreaterThanOrEqual("__ingest_ts", seven_days_ago),
    EqualTo("customer_id", "cus_11TqNF6bCebUjnx55pk6vs"),
    EqualTo("project_id", "proj_11STWG6AokEni2F3eQugHb")
)

# PyIceberg will use this to prune partitions/files and return only the matching rows
scan = table.scan(
    row_filter=filter_expr
)

# Get all matching files
files = []
for task in scan.plan_files():
    files.append({
        "file_path": task.file.file_path,
        "record_count": task.file.record_count,
        "file_size_in_bytes": task.file.file_size_in_bytes,
    })

print(f"Found {len(files)} files with {sum(f['record_count'] for f in files)} total records")
for f in files:
    print(f["file_path"])

Found 1 files with 28 total records
s3://unprice-lakehouse-dev/__r2_data_catalog/019c66ef-2bb2-7eb0-ba6f-40471a747007/019c80ad-4529-7ed2-9455-f5c36c5ed590/data/8eb4f53b-2923-4ca0-9789-48734a9b4f5f.parquet


In [226]:
from pyiceberg.expressions import GreaterThanOrEqual, EqualTo, And
from datetime import datetime, timedelta

# Calculate 7 days ago dynamically
seven_days_ago = (datetime.utcnow() - timedelta(days=7)).isoformat()

# Build the expression tree
filter_expr = And(
    GreaterThanOrEqual("__ingest_ts", seven_days_ago),
    EqualTo("customer_id", "cus_11TqNF6bCebUjnx55pk6vs")
)

# PyIceberg will use this to prune partitions/files and return only the matching rows
scan = table.scan(
    row_filter=filter_expr
)

arrow_table = scan.to_arrow()

pyarrow.Table
__ingest_ts: timestamp[us] not null
event_date: timestamp[us] not null
id: string not null
project_id: string not null
entitlement_id: string not null
feature_slug: string not null
customer_id: string not null
request_id: string not null
timestamp: timestamp[us] not null
schema_version: int32 not null
allowed: bool
denied_reason: string
latency: double
country: string
region: string
meta_id: string
action: string
key_id: string
usage: double
remaining: int64
unit_of_measure: string
cost: double
rate_amount: double
rate_currency: string
----
__ingest_ts: [[2026-02-21 15:15:32.000000,2026-02-21 15:15:32.000000,2026-02-21 15:15:32.000000,2026-02-21 15:15:32.000000,2026-02-21 15:15:32.000000,...,2026-02-21 15:15:32.000000,2026-02-21 15:15:32.000000,2026-02-21 15:15:32.000000,2026-02-21 15:15:32.000000,2026-02-21 15:15:32.000000]]
event_date: [[2026-02-21 00:00:00.000000,2026-02-21 00:00:00.000000,2026-02-21 00:00:00.000000,2026-02-21 00:00:00.000000,2026-02-21 00:00:00.000000

In [21]:
# 1. Create namespace
# catalog.create_namespace("default")

In [190]:
# 2. Load the pipeline-created table
# verifications = catalog.load_table("lakehouse.verification")
# print("Current Spec:", verifications.spec())
# metadata = catalog.load_table("lakehouse.metadata")
# print("Current Spec:", verifications.spec())
# entitlement_snapshot = catalog.load_table("lakehouse.entitlement_snapshot")
# print("Current Spec:", verifications.spec())
usage = catalog.load_table("default.usage3")
print("Current Spec:", usage.spec())
print(usage.metadata.location)

Current Spec: [
  1000: __ingest_ts_day: day(1)
  1001: project_partition: identity(5)
  1002: customer_partition: identity(6)
]
s3://sapohpta/__r2_data_catalog/019c8005-2735-7211-9fe1-0224dba747b7/019c8062-b4ce-7430-a948-5248d4602077


In [185]:
old_partition_name = usage.spec().fields[0].name 
print(old_partition_name)

__ingest_ts_day


In [169]:
# verifications
# Make sure to use the namespace defined in the pipeline (often 'default')
table = catalog.load_table("lakehouse.verification")

# Add your new partition keys
# This example adds 'region' and 'account_id' as identity partitions
with table.update_spec() as update:
    # A. Remove all existing partition fields
    for field in table.spec().fields:
        # We remove by the partition field name (not source column name)
        update.remove_field(field.name)

    update.add_field("event_date", DayTransform(), "date_partition")
    update.add_field("project_id", IdentityTransform(), "project_partition")
    update.add_field("customer_id", IdentityTransform(), "customer_partition")

print("New Spec:", table.spec())
print("\nPartition Data Preview:")
print(table.inspect.partitions().to_pandas())

NoSuchTableError: TableActionForbidden: Table not found or action can_get_metadata forbidden for Anonymous

In [187]:
# usage
# Make sure to use the namespace defined in the pipeline (often 'default')
table = catalog.load_table("default.usage3")

# Add your new partition keys
# This example adds 'region' and 'account_id' as identity partitions
with table.update_spec() as update:
    # Remove the existing ingestion-time partition
    # update.remove_field(old_partition_name)

    # update.add_field("event_date", DayTransform(), old_partition_name)
    update.add_field("project_id", IdentityTransform(), "project_partition")
    update.add_field("customer_id", IdentityTransform(), "customer_partition")

print("New Spec:", table.spec())
print("\nPartition Data Preview:")
print(table.inspect.partitions().to_pandas())


New Spec: [
  1000: __ingest_ts_day: day(1)
  1001: project_partition: identity(5)
  1002: customer_partition: identity(6)
]

Partition Data Preview:
                                           partition  spec_id  record_count  \
0  {'__ingest_ts_day': 2026-02-21, 'project_parti...        0            65   

   file_count  total_data_file_size_in_bytes  position_delete_record_count  \
0           1                          23438                             0   

   position_delete_file_count  equality_delete_record_count  \
0                           0                             0   

   equality_delete_file_count         last_updated_at  \
0                           0 2026-02-21 13:33:13.939   

   last_updated_snapshot_id  
0       2355266111474742490  


In [ ]:
# entitlement_snapshot
# Make sure to use the namespace defined in the pipeline (often 'default')
table = catalog.load_table("lakehouse.entitlement_snapshot")

# Add your new partition keys
# This example adds 'region' and 'account_id' as identity partitions
with table.update_spec() as update:
    # A. Remove all existing partition fields
    for field in table.spec().fields:
        # We remove by the partition field name (not source column name)
        update.remove_field(field.name)

    update.add_field("event_date", DayTransform(), "date_partition")
    update.add_field("project_id", IdentityTransform(), "project_partition")
    update.add_field("customer_id", IdentityTransform(), "customer_partition")

print("New Spec:", table.spec())
print("\nPartition Data Preview:")
print(table.inspect.partitions().to_pandas())

In [101]:
# metadata
# Make sure to use the namespace defined in the pipeline (often 'default')
table = catalog.load_table("lakehouse.metadata")

# Add your new partition keys
# This example adds 'region' and 'account_id' as identity partitions
with table.update_spec() as update:
    # A. Remove all existing partition fields
    for field in table.spec().fields:
        # We remove by the partition field name (not source column name)
        update.remove_field(field.name)

    update.add_field("event_date", DayTransform(), "date_partition")
    update.add_field("project_id", IdentityTransform(), "project_partition")
    update.add_field("customer_id", IdentityTransform(), "customer_partition")

print("New Spec:", table.spec())
print("\nPartition Data Preview:")
print(table.inspect.partitions().to_pandas())

New Spec: [
  1001: date_partition: day(3)
  1002: project_partition: identity(4)
  1003: customer_partition: identity(5)
]

Partition Data Preview:


ValueError: Cannot get a snapshot as the table does not have any.

In [102]:
# 4. Load the pipeline-created table
verifications = catalog.load_table("lakehouse.verification")
print("Current Spec:", verifications.spec())
metadata = catalog.load_table("lakehouse.metadata")
print("Current Spec:", verifications.spec())
entitlement_snapshot = catalog.load_table("lakehouse.entitlement_snapshot")
print("Current Spec:", verifications.spec())
usage = catalog.load_table("lakehouse.usage")
print("Current Spec:", verifications.spec())
print(verifications.metadata.location)

Current Spec: [
  1001: date_partition: day(3)
  1002: project_partition: identity(4)
  1003: customer_partition: identity(7)
]
Current Spec: [
  1001: date_partition: day(3)
  1002: project_partition: identity(4)
  1003: customer_partition: identity(7)
]
Current Spec: [
  1001: date_partition: day(3)
  1002: project_partition: identity(4)
  1003: customer_partition: identity(7)
]
Current Spec: [
  1001: date_partition: day(3)
  1002: project_partition: identity(4)
  1003: customer_partition: identity(7)
]
s3://unprice-dev/__r2_data_catalog/019c7d38-b6b3-7c90-94ee-da8a2e523815/019c7d3b-52be-7790-ba1a-c2abb7ca1803
